# Part 14: All-Models Ensemble Screening of COCONUT

Screens COCONUT with **every trained model at once** — RF (Part 6/8) + GCN
(Part 9) + GIN (Part 10) + AttentiveFP (Part 12) + Hybrid GCN+GAT (Part 13) —
and reports consensus + soft-vote ensemble hits for docking.

Design: read-only. This notebook only **loads** weights and metrics from
Parts 6/8/9/10/12/13 and writes its own outputs here. It never modifies
those folders. Models whose weights are absent (e.g. 12/13 before you train
them on Colab) are skipped automatically, and the vote runs on the rest.

In [ ]:
!git clone https://github.com/arjunpahi/Natural_MDM2_Inhibitor_Discovery_using_ML.git
%cd Natural_MDM2_Inhibitor_Discovery_using_ML

In [ ]:
!pip install torch-geometric rdkit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax, scatter
from rdkit import Chem
from rdkit.Chem import FilterCatalog, Descriptors
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Model architectures (same classes as Parts 9/10/12/13)

In [ ]:
class GCN(nn.Module):
    def __init__(self, num_node_features=78, hidden_dim=128, num_classes=2, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.dropout = dropout
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        return self.classifier(x)

In [ ]:
class GINEncoder(nn.Module):
    def __init__(self, num_node_features=78, hidden_dim=300, num_layers=5, dropout=0.2):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.gin_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()
        self.gin_layers.append(nn.Linear(num_node_features, hidden_dim))
        self.bn_layers.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            self.gin_layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.bn_layers.append(nn.BatchNorm1d(hidden_dim))
        self.eps = nn.ParameterList([nn.Parameter(torch.zeros(1)) for _ in range(num_layers)])

    def forward(self, x, edge_index, batch):
        for i in range(self.num_layers):
            src, dst = edge_index
            agg = torch.zeros_like(x)
            agg.scatter_add_(0, dst.unsqueeze(1).expand_as(x[src]), x[src])
            x = self.gin_layers[i]((1 + self.eps[i]) * x + agg)
            x = self.bn_layers[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)


class MDM2Classifier(nn.Module):
    def __init__(self, encoder, hidden_dim=600, num_classes=2):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes)
        )

    def forward(self, x, edge_index, batch):
        return self.classifier(self.encoder(x, edge_index, batch))

In [ ]:
class AttentiveLayer(nn.Module):
    def __init__(self, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.align = nn.Linear(3 * hidden_dim, 1)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)
        self.dropout = dropout
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, h, edge_index, edge_attr):
        src, dst = edge_index
        score = self.leaky(self.align(torch.cat([h[dst], h[src], edge_attr], dim=1))).squeeze(-1)
        alpha = softmax(score, dst)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        context = scatter(self.value(h[src]) * alpha.unsqueeze(-1), dst,
                          dim=0, dim_size=h.size(0), reduce='sum')
        return self.gru(F.elu(context), h)


class AttentiveReadout(nn.Module):
    def __init__(self, hidden_dim=128, timesteps=2, dropout=0.2):
        super().__init__()
        self.align = nn.Linear(2 * hidden_dim, 1)
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)
        self.timesteps = timesteps
        self.dropout = dropout
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, h, batch):
        q = global_mean_pool(h, batch)
        for _ in range(self.timesteps):
            score = self.leaky(self.align(torch.cat([q[batch], h], dim=1))).squeeze(-1)
            beta = softmax(score, batch)
            beta = F.dropout(beta, p=self.dropout, training=self.training)
            context = scatter(h * beta.unsqueeze(-1), batch,
                              dim=0, dim_size=q.size(0), reduce='sum')
            q = self.gru(F.elu(context), q)
        return q


class AttentiveFP(nn.Module):
    def __init__(self, num_node_features=78, num_edge_features=6, hidden_dim=128,
                 num_layers=2, num_timesteps=2, num_classes=2, dropout=0.2):
        super().__init__()
        self.atom_embed = nn.Linear(num_node_features, hidden_dim)
        self.edge_embed = nn.Linear(num_edge_features, hidden_dim)
        self.layers = nn.ModuleList([AttentiveLayer(hidden_dim, dropout) for _ in range(num_layers)])
        self.readout = AttentiveReadout(hidden_dim, num_timesteps, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x, edge_index, edge_attr, batch):
        h = F.relu(self.atom_embed(x))
        e = F.relu(self.edge_embed(edge_attr))
        for layer in self.layers:
            h = layer(h, edge_index, e)
        return self.classifier(self.readout(h, batch))

In [ ]:
class HybridGCNGAT(nn.Module):
    def __init__(self, num_node_features=78, gcn_hidden=128, gat_out=32, gat_heads=4,
                 num_classes=2, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, gcn_hidden)
        self.conv2 = GCNConv(gcn_hidden, gcn_hidden)
        self.bn1 = nn.BatchNorm1d(gcn_hidden)
        self.bn2 = nn.BatchNorm1d(gcn_hidden)
        self.gat = GATConv(gcn_hidden, gat_out, heads=gat_heads, dropout=dropout)
        self.dropout = dropout
        self.classifier = nn.Sequential(
            nn.Linear(gat_out * gat_heads * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat(x, edge_index))
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        return self.classifier(x)

## 2. Load trained weights (missing ones are skipped)

In [ ]:
# Load every trained model whose weights exist (missing ones are skipped).
# GCN + GIN ship in this repo; AttentiveFP/Hybrid appear after you train Parts 12/13 on Colab.
import os, urllib.request
BASE = 'https://raw.githubusercontent.com/Techbjd/new_method/main'
WANT = {
    'gcn': 'Part_9/gcn_scratch_model.pth',
    'gin': 'Part_10/pretrained_gin_mdm2.pth',
    'afp': 'Part_12/attentivefp_mdm2.pth',
    'hyb': 'Part_13/hybrid_gcn_gat_mdm2.pth',
}
MODELS, HAVE = {}, {}
for key, path in WANT.items():
    if not os.path.exists(path):
        try:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            urllib.request.urlretrieve(f'{BASE}/{path}', path)
            print(f'Downloaded {path}')
        except Exception as e:
            print(f'{path} not available ({e}). Train its Part first — skipping.')
            HAVE[key] = False
            continue
    HAVE[key] = True

def _load(model, path):
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model

try:
    if HAVE.get('gcn'):
        MODELS['gcn'] = _load(GCN().to(device), WANT['gcn'])
        print('GCN (Part 9) loaded.')
    if HAVE.get('gin'):
        MODELS['gin'] = _load(MDM2Classifier(GINEncoder().to(device)).to(device), WANT['gin'])
        print('GIN (Part 10) loaded.')
    if HAVE.get('afp'):
        MODELS['afp'] = _load(AttentiveFP().to(device), WANT['afp'])
        print('AttentiveFP (Part 12) loaded.')
    if HAVE.get('hyb'):
        MODELS['hyb'] = _load(HybridGCNGAT().to(device), WANT['hyb'])
        print('Hybrid GCN+GAT (Part 13) loaded.')
except Exception as e:
    print(f'Weight-shape mismatch, check the Part that trained it: {e}')
print(f'\nActive models: {sorted(MODELS)} (need at least 1)')

## 3. Load COCONUT data

In [ ]:
coconut = pd.read_csv('Part_8/screening_results.csv')
print(f"COCONUT compounds: {len(coconut)}")
coconut.head()

In [ ]:
# OPTION B — screen directly from the raw COCONUT database (paper Section 2.5).
# Same sequential filters as Part 11 (valid SMILES -> PAINS -> Brenk -> Ro5).
# Skip this cell to keep the Part_8 table loaded above.
import os
from rdkit.Chem import FilterCatalog, Descriptors
RAW_COCONUT = 'coconut_csv-03-2025.csv'
if not os.path.exists(RAW_COCONUT):
    print(f'{RAW_COCONUT} not found — keeping the Part_8 table. Nothing changed.')
else:
    raw = pd.read_csv(RAW_COCONUT, usecols=lambda c: c in (
        'identifier', 'id', 'ID', 'name', 'canonical_smiles', 'smiles', 'SMILES'))
    smi_col = next((c for c in ['canonical_smiles', 'smiles', 'SMILES'] if c in raw.columns), None)
    id_col = next((c for c in ['identifier', 'id', 'ID', 'name'] if c in raw.columns), None)
    _p = FilterCatalog.FilterCatalogParams()
    _p.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    _pcat = FilterCatalog.FilterCatalog(_p)
    _b = FilterCatalog.FilterCatalogParams()
    _b.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    _bcat = FilterCatalog.FilterCatalog(_b)
    keep_id, keep_smi = [], []
    n_inv = n_pains = n_brenk = n_ro5 = 0
    for _, r in tqdm(raw.iterrows(), total=len(raw), desc='COCONUT filtering'):
        mol = Chem.MolFromSmiles(r[smi_col])
        if mol is None:
            n_inv += 1
            continue
        if _pcat.HasMatch(mol):
            n_pains += 1
            continue
        if _bcat.HasMatch(mol):
            n_brenk += 1
            continue
        if not (Descriptors.ExactMolWt(mol) <= 500 and Descriptors.NumHAcceptors(mol) <= 10
                and Descriptors.NumHDonors(mol) <= 5 and Descriptors.MolLogP(mol) <= 5):
            n_ro5 += 1
            continue
        keep_id.append(r[id_col] if id_col else f'CNP{len(keep_id)}')
        keep_smi.append(r[smi_col])
    coconut = pd.DataFrame({'identifier': keep_id, 'canonical_smiles': keep_smi})
    print(f'Removed: {n_inv} invalid, {n_pains} PAINS, {n_brenk} Brenk, {n_ro5} Ro5-violating')
    print(f'Screenable COCONUT compounds: {len(coconut)}')

## 4. Featurize COCONUT SMILES → Graphs

In [ ]:
ATOM_CHOICES = {
    'atomic_num': list(range(1, 101)),
    'degree': [0, 1, 2, 3, 4, 5],
    'formal_charge': [-2, -1, 0, 1, 2, 3],
    'num_hs': [0, 1, 2, 3, 4],
    'hybridization': [
        Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2
    ]
}

def one_hot(val, choices):
    enc = [0] * len(choices)
    if val in choices:
        enc[choices.index(val)] = 1
    return enc

def atom_features(atom):
    f = []
    f += one_hot(atom.GetAtomicNum(), ATOM_CHOICES['atomic_num'])
    f += one_hot(atom.GetTotalDegree(), ATOM_CHOICES['degree'])
    f += one_hot(atom.GetFormalCharge(), ATOM_CHOICES['formal_charge'])
    f += one_hot(atom.GetTotalNumHs(), ATOM_CHOICES['num_hs'])
    f += one_hot(atom.GetHybridization(), ATOM_CHOICES['hybridization'])
    f.append(int(atom.GetIsAromatic()))
    f.append(int(atom.IsInRing()))
    return (f + [0] * 78)[:78]

def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.rdchem.BondType.SINGLE),
            int(bt == Chem.rdchem.BondType.DOUBLE),
            int(bt == Chem.rdchem.BondType.TRIPLE),
            int(bt == Chem.rdchem.BondType.AROMATIC),
            int(bond.GetIsConjugated()),
            int(bond.IsInRing())]

def mol_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    node_features = [atom_features(a) for a in mol.GetAtoms()]
    edge_index, edge_attr = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf = bond_features(b)
        edge_index.extend([[i, j], [j, i]])
        edge_attr.extend([bf, bf])
    if not edge_index:
        edge_index = [[0, 0]]
        edge_attr = [[0] * 6]
    return Data(
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float)
    )

In [ ]:
graphs = []
valid_idx = []
failed = 0

for idx, row in tqdm(coconut.iterrows(), total=len(coconut), desc="Featurizing"):
    g = mol_to_graph(row['canonical_smiles'])
    if g is not None:
        graphs.append(g)
        valid_idx.append(idx)
    else:
        failed += 1

print(f"Converted: {len(graphs)}/{len(coconut)} | Failed: {failed}")
screen_loader = DataLoader(graphs, batch_size=256, shuffle=False)
print(f"Batches: {len(screen_loader)}")

## 5. Screen with all loaded models

In [ ]:
# Score with every loaded model (each keeps its own forward signature).
PROBS, PREDS = {}, {}
with torch.no_grad():
    for key, model in MODELS.items():
        model.eval()
        probs, preds = [], []
        for batch in tqdm(screen_loader, desc=f"{key.upper()} screening"):
            batch = batch.to(device)
            if key == 'afp':
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            else:
                out = model(batch.x, batch.edge_index, batch.batch)
            p = F.softmax(out, dim=1)[:, 1]
            probs.extend(p.cpu().numpy())
            preds.extend(out.argmax(dim=1).cpu().numpy())
        PROBS[key] = np.array(probs)
        PREDS[key] = np.array(preds)
        print(f"{key.upper()}: active @0.5 = {(PREDS[key] == 1).sum()} / {len(PREDS[key])}")

## 6. Consensus + soft-vote ensemble

In [ ]:
results = coconut.iloc[valid_idx][['identifier', 'canonical_smiles']].copy()

# RF baseline when the Part 8 table is the input (optional, skipped in raw-COCONUT mode)
HAVE_RF = False
try:
    rf_orig = pd.read_csv('Part_8/screening_results.csv')
    if set(['prediction', 'prob_class_1']).issubset(rf_orig.columns) and len(rf_orig) == len(coconut):
        results['rf_prediction'] = rf_orig.iloc[valid_idx]['prediction'].values
        results['rf_prob_active'] = rf_orig.iloc[valid_idx]['prob_class_1'].values
        HAVE_RF = True
except Exception as e:
    print(f'RF baseline skipped ({e}).')

for key in MODELS:
    results[f'{key}_prediction'] = PREDS[key]
    results[f'{key}_prob'] = PROBS[key]

# Majority consensus over every available voter (RF + loaded GNNs)
vote_cols = [c for c in results.columns if c.endswith('_prediction')]
results['consensus'] = (results[vote_cols].sum(axis=1) >= (len(vote_cols) // 2 + 1)).astype(int)
# Soft vote: mean probability over every available scorer
prob_cols = [c for c in results.columns if c.endswith('_prob')]
results['ensemble_prob'] = results[prob_cols].mean(axis=1)

print(f"Total screened: {len(results)} | voters: {vote_cols}")
print(f"Consensus active: {results['consensus'].sum()}")
print(f"Ensemble prob > 0.6: {(results['ensemble_prob'] > 0.6).sum()}")
results.head(10)

## 7. Medchem gate + strict hits

In [ ]:
# PAINS/Brenk/Ro5 gate for the final shortlist (same filters as Part 11).
from rdkit.Chem import FilterCatalog, Descriptors
_p = FilterCatalog.FilterCatalogParams()
_p.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
_pcat = FilterCatalog.FilterCatalog(_p)
_b = FilterCatalog.FilterCatalogParams()
_b.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
_bcat = FilterCatalog.FilterCatalog(_b)

def medchem_ok(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    if _pcat.HasMatch(mol) or _bcat.HasMatch(mol):
        return False
    return (Descriptors.ExactMolWt(mol) <= 500 and Descriptors.NumHAcceptors(mol) <= 10
            and Descriptors.NumHDonors(mol) <= 5 and Descriptors.MolLogP(mol) <= 5)

results['medchem_pass'] = [medchem_ok(s) for s in tqdm(results['canonical_smiles'], desc='Medchem filter')]
print(f"Medchem pass: {results['medchem_pass'].sum()} / {len(results)}")

In [ ]:
results.to_csv('ensemble_screening_results.csv', index=False)

strict = results[(results['consensus'] == 1) & (results['medchem_pass'])].copy()
strict = strict.sort_values('ensemble_prob', ascending=False)
strict.to_csv('ensemble_consensus_hits.csv', index=False)
print(f'Saved ensemble_screening_results.csv ({len(results)} rows)')
print(f'Saved {len(strict)} strict consensus hits to ensemble_consensus_hits.csv')

for t in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9]:
    m = (results['ensemble_prob'] > t) & (results['medchem_pass'])
    print(f'  ensemble_prob > {t} (+medchem): {m.sum()} hits')

try:
    rf116 = set(pd.read_csv('Part_8/filtered_compounds.csv')['identifier'].astype(str))
    s = set(strict['identifier'].astype(str))
    print(f'Overlap with paper RF-116: {len(rf116 & s)} / 116 confirmed')
    print(f'New candidates beyond RF-116: {len(s - rf116)}')
except Exception as e:
    print(f'(RF-116 overlap skipped: {e})')
strict.head(10)

## 8. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for key in MODELS:
    axes[0].hist(PROBS[key], bins=50, alpha=0.5, label=key.upper())
axes[0].axvline(x=0.85, color='red', linestyle='--', linewidth=2, label='Strict 0.85')
axes[0].set_xlabel('P(active)'); axes[0].set_ylabel('Count')
axes[0].set_title('All-model probability distributions (COCONUT)'); axes[0].legend()
axes[1].hist(results['ensemble_prob'], bins=50, color='teal', edgecolor='black')
axes[1].axvline(x=0.6, color='red', linestyle='--', linewidth=2, label='Paper-style 0.6')
axes[1].set_xlabel('Ensemble P(active)'); axes[1].set_title('Soft-vote ensemble distribution'); axes[1].legend()
plt.tight_layout()
plt.savefig('ensemble_screening_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved ensemble_screening_analysis.png')

## 9. SDF for docking

In [ ]:
# SDF extraction of strict hits for docking (mirrors Part 8/11, guarded).
import os
SDF_PATH = ''  # e.g. 'coconut_sdf_3d-04-2025.sdf' — set path, then run
if 'strict' not in dir():
    print('No hits table yet. Run the cells above first.')
elif not SDF_PATH or not os.path.exists(SDF_PATH):
    print('Set SDF_PATH to the COCONUT SDF file to extract 3D structures for docking.')
    print(f'(Would extract {len(strict)} strict consensus hits.)')
else:
    target_ids = set(strict['identifier'].dropna().astype(str))
    supplier = Chem.SDMolSupplier(SDF_PATH)
    writer = Chem.SDWriter('ensemble_hits_for_docking.sdf')
    count = 0
    for mol in supplier:
        if mol is None:
            continue
        try:
            if mol.GetProp('IDENTIFIER') in target_ids:
                writer.write(mol)
                count += 1
        except KeyError:
            continue
    writer.close()
    print(f'Extracted {count} structures to ensemble_hits_for_docking.sdf')

In [ ]:
print("\n=== Part 14 Complete ===")
print("Outputs:")
print("  - ensemble_screening_results.csv (all compounds, all model scores)")
print("  - ensemble_consensus_hits.csv (strict consensus + medchem hits)")
print("  - ensemble_screening_analysis.png")
print("Parts 9/10/12/13 folders were only read, never modified.")